# Identidade processual e classe canônica

Análise exploratória, reprodutível e somente leitura. Nenhuma extração abaixo constitui componente de produção.

# 1. Setup

In [ ]:
from __future__ import annotations

import hashlib
import re

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from bracis_jusbrasil.database import connect_database, get_database_path

DATABASE_PATH = get_database_path()
connection = connect_database(DATABASE_PATH)
TRIBUNAIS = ('STF', 'STJ', 'TSE', 'TST', 'STM')
print({'database': str(DATABASE_PATH), 'modo': 'read-only'})

In [ ]:
def query_frame(sql: str, params: tuple[object, ...] = ()) -> pd.DataFrame:
    return pd.read_sql_query(sql, connection, params=params)


def preview(value: str, limit: int = 360) -> str:
    compact = ' '.join(value.split())
    return compact[:limit] + ('…' if len(compact) > limit else '')


def normalize_number(value: str | None) -> str | None:
    if not value:
        return None
    digits = re.sub(r'[^0-9]', '', value)
    return digits or None


def normalize_class(value: str | None) -> str | None:
    if not value:
        return None
    return re.sub(r'[^A-Z0-9]+', ' ', value.upper()).strip() or None

# 2. O conceito de identidade principal

**Número qualquer** é uma referência citada no texto. **Número principal** pertence ao acórdão representado pelo registro. Da mesma forma, uma classe só é principal quando ocorre no bloco estrutural associado a esse número; proximidade solta no corpo não é evidência suficiente.

# 3. Anatomia da identidade por tribunal

In [ ]:
acordaos = query_frame(
    "SELECT documento_id, id, tribunal, ano, relator, texto FROM documentos WHERE natureza = 'acordao' ORDER BY documento_id"
)
por_tribunal = acordaos.groupby('tribunal', as_index=False).agg(registros=('id', 'size')).sort_values('tribunal')
assert len(acordaos) == 1_000
assert por_tribunal['registros'].eq(200).all()
display(por_tribunal)

amostras = acordaos.groupby('tribunal', group_keys=False).head(2)[['tribunal', 'documento_id', 'texto']].copy()
amostras['início'] = amostras.pop('texto').map(lambda value: preview(value, 850))
display(amostras)

anatomia = pd.DataFrame([
    ('STF', 'cabeçalho inicial após data/turma', 'decimal ou CNJ', 'antes do número', 'topônimo frequente', 'referências a outros feitos podem aparecer logo depois'),
    ('STJ', 'primeiro título do documento', 'decimal com UF', 'antes de Nº', 'sufixo/partes', 'número interno pode coexistir'),
    ('TSE', 'título após ACÓRDÃO', 'CNJ eleitoral', 'antes de Nº', 'cidade/estado', 'classe pode ser composta'),
    ('TST', 'bloco de vistos e relato', 'CNJ com prefixo', 'prefixo do número', 'no próprio número', 'cabeçalho pode vir após ementa'),
    ('STM', 'ata/cabeçalho inicial', 'CNJ militar', 'antes de Nº', 'sufixo /UF', 'ata menciona recursos anteriores'),
], columns=['tribunal', 'bloco de identidade', 'número principal', 'classe', 'UF', 'particularidades'])
display(anatomia)

# 4. Número principal

In [ ]:
CNJ_RE = re.compile(r'(?<!\d)\d{3,7}-\s*\d{2}\.\d{4}\.\d\.\d{2}\.\d{4}(?:/[A-Z]{2})?(?!\d)', re.I)
DECIMAL_RE = re.compile(r'(?<!\d)\d{1,3}(?:[.,]\d{3})+(?:-\d+)?(?!\d)')
TST_RE = re.compile(r'(?<![A-Z0-9])(?:TST-)?[A-Z]{1,6}(?:-[A-Z]{1,6}){1,5}-\d{3,7}-\d{2}\.\d{4}\.\d\.\d{2}\.\d{4}(?!\d)', re.I)
UF_RE = re.compile(r'(?:[-/]\s*|\b)(AC|AL|AP|AM|BA|CE|DF|ES|GO|MA|MT|MS|MG|PA|PB|PR|PE|PI|RJ|RN|RS|RO|RR|SC|SP|SE|TO)\b')
BLOCK_LIMITS = {'STF': 1_000, 'STJ': 900, 'TSE': 1_100, 'TST': 3_000, 'STM': 1_200}


def identity_block(text: str, tribunal: str) -> tuple[str, int, str]:
    limit = BLOCK_LIMITS[tribunal]
    return text[:limit], 0, f'cabeçalho_{tribunal.lower()}_{limit}'


def number_candidates(block: str, tribunal: str) -> list[dict[str, object]]:
    patterns = [('cnj', CNJ_RE), ('decimal', DECIMAL_RE)]
    if tribunal == 'TST':
        patterns.insert(0, ('tst_cnj', TST_RE))
    found: list[dict[str, object]] = []
    seen: set[tuple[int, int]] = set()
    for name, pattern in patterns:
        for match in pattern.finditer(block):
            span = match.span()
            if span in seen:
                continue
            seen.add(span)
            before = block[max(0, match.start() - 140):match.start()].upper()
            score = 10 + max(0, 4 - match.start() / 350)
            if re.search(r'N[ºO°]\s*$', before):
                score += 8
            if name == 'tst_cnj':
                score += 8
            if name == 'cnj' and tribunal in {'TSE', 'STM'}:
                score += 5
            if name == 'decimal' and tribunal in {'STF', 'STJ'}:
                score += 4
            found.append({'raw': match.group(0), 'position': match.start(), 'pattern': name, 'score': score})
    return sorted(found, key=lambda item: (-float(item['score']), int(item['position'])))


def extract_main_number(row: pd.Series) -> dict[str, object]:
    block, offset, block_pattern = identity_block(row['texto'], row['tribunal'])
    candidates = number_candidates(block, row['tribunal'])
    if not candidates:
        return {'identity_block': block, 'block_offset': offset, 'block_pattern': block_pattern, 'numero_principal_raw': None, 'numero_principal_normalizado': None, 'numero_posicao': None, 'numero_source_pattern': None, 'numero_status': 'não extraído', 'numero_ambiguo': False, 'uf': None}
    selected = candidates[0]
    ambiguous = len(candidates) > 1 and float(candidates[0]['score']) == float(candidates[1]['score'])
    after = block[int(selected['position']):int(selected['position']) + 180]
    uf_match = UF_RE.search(after.upper())
    return {'identity_block': block, 'block_offset': offset, 'block_pattern': block_pattern, 'numero_principal_raw': selected['raw'], 'numero_principal_normalizado': normalize_number(str(selected['raw'])), 'numero_posicao': int(selected['position']) + offset, 'numero_source_pattern': selected['pattern'], 'numero_status': 'ambíguo' if ambiguous else 'extraído', 'numero_ambiguo': ambiguous, 'uf': uf_match.group(1) if uf_match else None}

identidades = acordaos.join(pd.DataFrame([extract_main_number(row) for _, row in acordaos.iterrows()]))
print('Parsing experimental: o candidato de maior evidência no bloco não é uma decisão de identidade final.')

In [ ]:
numero_metricas = identidades.assign(
    extraido=identidades['numero_status'].eq('extraído'),
    nao_extraido=identidades['numero_status'].eq('não extraído'),
    ambiguo=identidades['numero_status'].eq('ambíguo'),
).groupby('tribunal', as_index=False).agg(registros=('id', 'size'), **{'número principal extraído': ('extraido', 'sum'), 'não extraído': ('nao_extraido', 'sum'), 'ambíguo': ('ambiguo', 'sum')})
display(numero_metricas)
display(identidades.groupby('numero_status', group_keys=False).head(5)[['tribunal', 'documento_id', 'numero_principal_raw', 'numero_source_pattern', 'numero_posicao']])

ax = numero_metricas.plot.bar(x='tribunal', y=['número principal extraído', 'não extraído', 'ambíguo'], figsize=(8, 4), rot=0)
ax.set_title('Cobertura de número principal por tribunal')
ax.set_xlabel('tribunal')
ax.set_ylabel('acórdãos')
ax.figure.tight_layout()
plt.show()

qualidade_numero = identidades.assign(qualidade=identidades['numero_status']).pivot_table(index='tribunal', columns='qualidade', values='id', aggfunc='count', fill_value=0)
ax = qualidade_numero.plot.bar(stacked=True, figsize=(8, 4), rot=0)
ax.set_title('Qualidade do número principal por tribunal')
ax.set_xlabel('tribunal')
ax.set_ylabel('acórdãos')
ax.figure.tight_layout()
plt.show()

# 5. Classe associada ao número principal

In [ ]:
def extract_main_class(row: pd.Series) -> dict[str, object]:
    if pd.isna(row['numero_posicao']):
        return {'classe_raw': None, 'classe_normalizada': None, 'classe_posicao': None, 'classe_distancia': None}
    local_position = int(row['numero_posicao']) - int(row['block_offset'])
    before = row['identity_block'][max(0, local_position - 180):local_position]
    upper = before.upper()
    marker = re.search(r'([A-ZÀ-Ú][A-ZÀ-Ú .-]{2,}?)\s+N[ºO°]\s*$', upper)
    if marker:
        raw = marker.group(1).strip(' .-')[-90:]
        for separator in (' ACÓRDÃO ', ' TURMA '):
            if separator in raw:
                raw = raw.rsplit(separator, 1)[-1]
        position = int(row['numero_posicao']) - len(before) + marker.start(1)
    elif row['tribunal'] == 'TST' and isinstance(row['numero_principal_raw'], str):
        prefix = row['numero_principal_raw'].rsplit('-', 2)[0].replace('TST-', '')
        raw = prefix.replace('-', ' ')
        position = int(row['numero_posicao'])
    else:
        raw, position = None, None
    distance = int(row['numero_posicao']) - position if position is not None else None
    return {'classe_raw': raw, 'classe_normalizada': normalize_class(raw), 'classe_posicao': position, 'classe_distancia': distance}

identidades = identidades.join(pd.DataFrame([extract_main_class(row) for _, row in identidades.iterrows()]))
display(identidades[['tribunal', 'documento_id', 'classe_raw', 'numero_principal_raw', 'classe_distancia']].groupby('tribunal', group_keys=False).head(3))

# 6. Auditoria de qualidade da classe

In [ ]:
def audit_class(row: pd.Series) -> tuple[str, str]:
    value = row['classe_raw']
    if not isinstance(value, str) or not value.strip():
        return 'ausente', 'sem classe no bloco do número'
    upper = value.upper()
    signals = []
    if len(value) > 70: signals.append('extensa')
    if 'ART' in upper: signals.append('artigo')
    if re.search(r'\d{2}[/.]\d{2}', upper): signals.append('data')
    if any(token in upper for token in ('EMENTA', 'RELATOR', 'ADVOGADO', 'RECORRENTE')): signals.append('campo narrativo')
    if row['classe_distancia'] is not None and row['classe_distancia'] > 150: signals.append('distante')
    if len(re.findall(r'[A-ZÀ-Ú]{2,}', upper)) > 12: signals.append('muitos termos')
    return ('suspeita', ', '.join(signals)) if signals else ('confiável', 'bloco e forma compacta')

identidades[['classe_status', 'classe_motivo']] = identidades.apply(audit_class, axis=1, result_type='expand')
classe_metricas = identidades.assign(
    numero_principal=identidades['numero_status'].eq('extraído'),
    classe_confiavel=identidades['classe_status'].eq('confiável'),
    classe_ausente=identidades['classe_status'].eq('ausente'),
    classe_ambigua=identidades['classe_status'].eq('suspeita'),
).groupby('tribunal', as_index=False).agg(**{'número principal': ('numero_principal', 'sum'), 'classe confiável': ('classe_confiavel', 'sum'), 'classe ausente': ('classe_ausente', 'sum'), 'classe ambígua': ('classe_ambigua', 'sum')})
display(classe_metricas)
display(identidades[identidades['classe_status'].eq('confiável')][['tribunal', 'classe_raw', 'numero_principal_raw']].head(12))
display(identidades[identidades['classe_status'].eq('suspeita')][['tribunal', 'classe_raw', 'classe_motivo', 'numero_principal_raw']].head(12))

ax = identidades.pivot_table(index='tribunal', columns='classe_status', values='id', aggfunc='count', fill_value=0).plot.bar(stacked=True, figsize=(8, 4), rot=0)
ax.set_title('Classes confiáveis e suspeitas por tribunal')
ax.set_xlabel('tribunal')
ax.set_ylabel('acórdãos')
ax.figure.tight_layout()
plt.show()

# 7. Taxonomia observada de classes

In [ ]:
classes_confiaveis = identidades[identidades['classe_status'].eq('confiável')].copy()
taxonomia = (classes_confiaveis.groupby(['tribunal', 'classe_raw', 'classe_normalizada'], as_index=False)
    .agg(frequencia=('id', 'size'), exemplos=('documento_id', lambda values: ', '.join(values.head(3))))
    .sort_values(['frequencia', 'tribunal'], ascending=[False, True]))
display(taxonomia.head(30))

top_classes = taxonomia.head(12).sort_values('frequencia')
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(top_classes['classe_raw'], top_classes['frequencia'], color='#4472c4')
ax.set_title('Taxonomia observada de classes confiáveis')
ax.set_xlabel('frequência')
ax.set_ylabel('classe bruta')
fig.tight_layout()
plt.show()

In [ ]:
def normalize_class(value: str | None) -> str | None:
    if pd.isna(value) or not value:
        return None
    return re.sub(r'[^A-Z0-9]+', ' ', str(value).upper()).strip() or None

# 8. Classe-base e modificadores

In [ ]:
def split_class(value: str | None) -> tuple[str | None, str | None, str]:
    if pd.isna(value) or not value:
        return None, None, 'ausente'
    normalized = normalize_class(value)
    if ' NO ' in normalized:
        modifier, base = normalized.rsplit(' NO ', 1)
        return base, modifier, 'composta'
    return normalized, None, 'simples'

identidades[['classe_base', 'modificadores', 'forma_classe']] = identidades['classe_normalizada'].map(split_class).apply(pd.Series)
formas = classes_confiaveis.assign(forma_classe=classes_confiaveis['classe_normalizada'].map(lambda value: split_class(value)[2])).groupby('forma_classe', as_index=False).agg(registros=('id', 'size'))
display(formas)

ax = formas.plot.bar(x='forma_classe', y='registros', legend=False, figsize=(6, 4), rot=0, color='#70ad47')
ax.set_title('Classes simples e compostas')
ax.set_xlabel('forma observada')
ax.set_ylabel('registros confiáveis')
ax.figure.tight_layout()
plt.show()

mesmo_numero = identidades[identidades['classe_status'].eq('confiável')].groupby(['tribunal', 'numero_principal_normalizado'], dropna=True).agg(classes=('classe_normalizada', 'nunique'), bases=('classe_base', 'nunique')).reset_index()
display(mesmo_numero.query('classes > 1').head(15))

# 9. Os 14 casos sem número

A seção revisita todos os casos que permanecem sem número principal no experimento atual; a referência aos 14 casos é o ponto de partida do Notebook 02, não uma suposição sobre o novo método.

In [ ]:
sem_numero = identidades[identidades['numero_status'].eq('não extraído')].copy()
sem_numero['causa'] = sem_numero['tribunal'].map(lambda tribunal: f'nenhum candidato no {BLOCK_LIMITS[tribunal]} caracteres do bloco')
sem_numero['número existe?'] = sem_numero['texto'].map(lambda value: bool(CNJ_RE.search(value) or DECIMAL_RE.search(value) or TST_RE.search(value)))
sem_numero['parser melhorável?'] = sem_numero['número existe?']
sem_numero['FTS ajuda?'] = 'a testar com candidato observável'
sem_numero['início do texto'] = sem_numero['texto'].map(lambda value: preview(value, 400))
display(sem_numero[['documento_id', 'tribunal', 'causa', 'número existe?', 'parser melhorável?', 'FTS ajuda?', 'início do texto']])

# 10. FTS como fallback real

FTS recupera candidatos textuais depois de uma falha ou ambiguidade estrutural; um retorno só é contado quando sua ocorrência permanece no bloco de identidade do próprio documento.

In [ ]:
def fts_candidates(query: str, occurrence: str, limit: int = 10) -> pd.DataFrame:
    try:
        rows = query_frame(
            '''SELECT d.documento_id, d.id, d.tribunal, d.texto
               FROM documentos_fts JOIN documentos AS d ON d.rowid = documentos_fts.rowid
               WHERE documentos_fts MATCH ? LIMIT ?''',
            (query, limit),
        )
    except Exception:
        return pd.DataFrame(columns=['documento_id', 'id', 'tribunal', 'posição', 'início', 'contexto'])
    rows['posição'] = rows['texto'].map(lambda value: value.casefold().find(occurrence.casefold()))
    rows['início'] = rows['texto'].map(lambda value: preview(value, 260))
    rows['contexto'] = rows.apply(lambda row: preview(row['texto'][max(0, row['posição'] - 80):row['posição'] + 180], 280), axis=1)
    return rows.drop(columns='texto')

fts_origem = identidades[identidades['numero_status'].ne('extraído')].dropna(subset=['numero_principal_raw']).head(5)
fts_resultados = []
for _, row in fts_origem.iterrows():
    found = fts_candidates(f'"{row["numero_principal_raw"]}"', row['numero_principal_raw'])
    if not found.empty:
        found['origem_documento_id'] = row['documento_id']
        found['revalidado_no_bloco'] = found['posição'].between(0, BLOCK_LIMITS[row['tribunal']])
        fts_resultados.append(found)
fts_recovery = pd.concat(fts_resultados, ignore_index=True) if fts_resultados else pd.DataFrame(columns=['revalidado_no_bloco'])
display(fts_recovery.head(20))
fts_metricas = pd.DataFrame({'método': ['parser estrutural', 'candidatos FTS revalidados'], 'registros': [int(identidades['numero_status'].eq('extraído').sum()), int(fts_recovery.get('revalidado_no_bloco', pd.Series(dtype=bool)).sum())]})
ax = fts_metricas.plot.bar(x='método', y='registros', legend=False, figsize=(7, 4), rot=0, color=['#4472c4', '#ed7d31'])
ax.set_title('Cobertura estrutural e candidatos FTS revalidados')
ax.set_xlabel('método')
ax.set_ylabel('registros')
ax.figure.tight_layout()
plt.show()

# 11. Recalcular as chaves candidatas

In [ ]:
elegiveis = identidades[identidades['numero_status'].eq('extraído') & identidades['classe_status'].eq('confiável')].copy()
elegiveis['chave_a'] = elegiveis['tribunal'] + '|' + elegiveis['numero_principal_normalizado']
elegiveis['chave_b'] = elegiveis['chave_a'] + '|' + elegiveis['classe_normalizada']
elegiveis['chave_c'] = elegiveis['chave_b'] + '|' + elegiveis['uf'].fillna('<sem_uf>')
elegiveis['chave_d'] = elegiveis['tribunal'] + '|' + elegiveis['classe_base'].fillna('<sem_base>') + '|' + elegiveis['numero_principal_normalizado']

resumo = []
for chave in ('chave_a', 'chave_b', 'chave_c', 'chave_d'):
    groups = elegiveis.groupby(chave)['id'].nunique()
    multi = groups[groups > 1]
    resumo.append({'chave': chave, 'registros elegíveis': len(elegiveis), 'grupos': len(groups), 'grupos multi-ID': len(multi), 'maior grupo': int(groups.max()) if len(groups) else 0, 'colisões': len(multi), 'registros sem chave': len(identidades) - len(elegiveis)})
resumo_chaves = pd.DataFrame(resumo)
display(resumo_chaves)

ax = resumo_chaves.plot.bar(x='chave', y=['grupos', 'grupos multi-ID'], figsize=(8, 4), rot=0)
ax.set_title('Colisões das chaves candidatas')
ax.set_xlabel('chave experimental')
ax.set_ylabel('grupos')
ax.figure.tight_layout()
plt.show()

# 12. Inspeção manual das colisões

In [ ]:
elegiveis['texto_hash'] = elegiveis['texto'].map(lambda value: hashlib.sha256(value.encode('utf-8')).hexdigest())
colisoes = elegiveis.groupby('chave_b').filter(lambda group: group['id'].nunique() > 1).copy()

def collision_label(group: pd.DataFrame) -> str:
    if group['texto_hash'].nunique() == 1:
        return 'provável mesmo feito'
    if group['classe_base'].nunique() == 1:
        return 'mesmo número com decisões/recursos diferentes'
    if group['numero_principal_normalizado'].nunique() > 1:
        return 'provavelmente feitos distintos'
    return 'indeterminado'

inspecao_colisoes = []
for key, group in colisoes.groupby('chave_b'):
    inspecao_colisoes.append({'chave': key, 'ids': ', '.join(sorted(group['id'].astype(str).unique())), 'tribunal': group['tribunal'].iloc[0], 'classe': group['classe_raw'].iloc[0], 'número': group['numero_principal_raw'].iloc[0], 'UF': group['uf'].iloc[0], 'texto idêntico': group['texto_hash'].nunique() == 1, 'relação provável': collision_label(group), 'cabeçalho': preview(group['texto'].iloc[0], 280)})
inspecao_colisoes = pd.DataFrame(inspecao_colisoes)
display(inspecao_colisoes.head(20))

classificacao = inspecao_colisoes['relação provável'].value_counts().rename_axis('classificação').reset_index(name='grupos') if not inspecao_colisoes.empty else pd.DataFrame({'classificação': ['sem grupos'], 'grupos': [0]})
ax = classificacao.plot.bar(x='classificação', y='grupos', legend=False, figsize=(9, 4), rot=20, color='#5b9bd5')
ax.set_title('Classificação exploratória das colisões')
ax.set_xlabel('leitura baseada apenas no corpus')
ax.set_ylabel('grupos')
ax.figure.tight_layout()
plt.show()

# 13. Definição operacional de feito

A decisão futura deve partir das colisões elegíveis: tribunal e número são o ponto de entrada; classe completa é evidência de desambiguação, classe-base é hipótese comparativa e UF só permanece quando adiciona informação observável. Modificadores merecem preservar a relação entre decisões do mesmo número, e IDs canônicos devem continuar como conjunto até validação posterior.

# 14. Hipótese de estrutura canônica

Uma futura estrutura conceitual pode preservar `tribunal`, número bruto/normalizado, classe bruta/normalizada, classe-base, modificadores, UF, conjunto de IDs e qualidade de parsing. Estes são campos de observação do notebook; nenhuma classe é criada no pacote.

# 15. Critérios de confiança do parsing

- **high confidence**: número e classe compacta no bloco esperado, sem candidato concorrente.
- **medium confidence**: número no bloco, classe ausente ou sinal leve de dúvida.
- **low confidence**: múltiplos candidatos, classe suspeita ou recuperação textual a revalidar.
- **unparsed**: nenhum número principal localizável no bloco.

São categorias operacionais, não probabilidades calibradas.

# 16. Findings

## Fatos confirmados

As tabelas e gráficos mostram cobertura por tribunal, qualidade de classe, falhas e colisões somente no subconjunto elegível.

## Hipótese de identidade

Número principal é indispensável; tribunal, classe e UF são avaliados como componentes distintos e não como uma concatenação presumida.

## Papel da classe, UF e FTS

Classe é evidência quando confiável e pode exigir separação entre base e modificadores. UF é contextual. FTS é recuperação de candidatos, nunca decisão isolada de identidade.

In [ ]:
print({'acórdãos': len(acordaos), 'números extraídos': int(identidades['numero_status'].eq('extraído').sum()), 'classes confiáveis': int(identidades['classe_status'].eq('confiável').sum())})
connection.close()
print('Conexão read-only encerrada.')